# 01 — Çalışma ortamı, SST-2 veri seti ve FP32 baz model (G2–G3)

**Proje:** Uç Cihazlar İçin FPGA Tabanlı BERT-Tiny Hızlandırıcısı

Bu not defteri şunları yapar:
1. Colab ortamını kurar (PyTorch, Transformers, Datasets)
2. SST-2 veri setini indirir ve inceler (G2)
3. BERT-Tiny modelini SST-2 üzerinde eğitir ve 32 bitlik doğruluğunu ölçer (G3)
4. Sonuçları Google Drive'a kaydeder

> Başlamadan önce: **Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU** seçin. (CPU'da da çalışır, yalnızca daha yavaş olur.)

## 1. Ortam kurulumu

In [ ]:
# Proje deposu
REPO_URL = "https://github.com/rumeysakolip/bert-tiny-fpga.git"

import os
if not os.path.exists("bert-tiny-fpga"):
    !git clone {REPO_URL}
%cd bert-tiny-fpga
!pip install -q -r requirements.txt

In [ ]:
import torch, transformers, datasets, platform
print("Python      :", platform.python_version())
print("PyTorch     :", torch.__version__)
print("Transformers:", transformers.__version__)
print("Datasets    :", datasets.__version__)
print("GPU         :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "yok (CPU)")

## 2. SST-2 veri setinin indirilmesi ve incelenmesi (G2)

In [ ]:
from datasets import load_dataset
sst2 = load_dataset("nyu-mll/glue", "sst2")
print(sst2)
for split in ["train", "validation"]:
    labels = sst2[split]["label"]
    print(f"{split:10s}: {len(labels):6d} cümle | olumlu oranı: %{100*sum(labels)/len(labels):.1f}")
print("\nÖrnekler:")
for ex in sst2["validation"].select(range(5)):
    print(("OLUMLU " if ex["label"] else "OLUMSUZ"), "|", ex["sentence"])

In [ ]:
# Model ve tokenizer'ın indirilmesi + mimari özeti
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import numpy as np
tok = AutoTokenizer.from_pretrained("prajjwal1/bert-tiny")
m = AutoModelForSequenceClassification.from_pretrained("prajjwal1/bert-tiny", num_labels=2)
print(m.config)
print(f"Parametre sayısı: {sum(p.numel() for p in m.parameters()):,}")

lens = [len(tok(s)["input_ids"]) for s in sst2["train"]["sentence"]]
print(f"Eğitim kümesi token uzunluğu: ort={np.mean(lens):.1f}, %99'luk dilim={np.percentile(lens, 99):.0f}, maks={max(lens)}")

## 3. FP32 baz modelin eğitimi (G3)

Varsayılan ayarlar: 4 epoch, batch 32, öğrenme oranı 1e-4, sabit dizi uzunluğu 64.
Turc ve ark. (2019) bu ayar aralığıyla BERT-Tiny için SST-2'de %83,2 (test) bildirmiştir.

In [ ]:
!python yazilim/egit_fp32.py --out_dir deneyler/fp32_baz

In [ ]:
import json
r = json.load(open("deneyler/fp32_baz/sonuclar.json"))
print(f"FP32 doğrulama doğruluğu: %{100*r['dogruluk']:.2f}  (en iyi epoch {r['en_iyi_epoch']})")
print(f"Parametre: {r['parametre_sayisi']:,}  |  FP32 boyut: {r['fp32_boyut_MB']} MB  |  Eğitim süresi: {r['egitim_suresi_sn']} sn")
for h in r["gecmis"]:
    print(h)

### (İsteğe bağlı) Öğrenme oranı taraması
Turc ve ark. 4 öğrenme oranını denemiştir. Zaman varsa en iyisini seçmek için:

In [ ]:
# for lr in ["3e-4", "1e-4", "5e-5"]:
#     !python yazilim/egit_fp32.py --lr {lr} --out_dir deneyler/fp32_lr{lr}

## 4. Sonuçların Google Drive'a kaydedilmesi

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/bert-tiny-fpga
!cp -r deneyler /content/drive/MyDrive/bert-tiny-fpga/
print("Kaydedildi: MyDrive/bert-tiny-fpga/deneyler")